# Shared data, inference and persistent cache

We restore the saved partitions and compute model outputs once. Subsequent parts load tables without rereading spectra or running the networks.

## Introduction

The analysis supports the selection of heads for subsequent experiments. We keep prediction quality, reconstruction and geometry separate. Definitions and limitations are described in [METHODOLOGY.md](METHODOLOGY.md), and execution instructions in [README.md](README.md).

### Assumptions

The evaluated pixels come from the saved partitions of one campaign; the result does not measure transfer to a new patient or acquisition. A positive annotation is an observed P; a missing annotation does not confirm the absence of a molecule. The unit of model comparison is the seed, not the pixel; duplicate tasks for the same condition and seed do not increase the number of repetitions. Candidate selection uses validation, while test provides subsequent confirmation.

### Notation

$X \in \mathbb{R}^{N\times M}$ denotes TIC-normalized spectra, $\hat X$ reconstructions, $Y \in \{0,1\}^{N\times C}$ annotations, and $A$ label availability. $Z=\mathrm{Enc}(X)$ is the latent representation used by the head and decoder; $U=(Z-\beta)/\gamma$ reverses the affine transformation of the final LayerNorm. Evidence states follow N=0, P=1, U=2; **state U and representation $U$ are different objects**. $s$ denotes the positive ranking score. $r$ is an independent training repetition.


In [1]:
import os
from pathlib import Path

import pandas as pd
from IPython.display import display

from msi_autoencoder_wrapper.analysis.autoencoder.experiments import predictive_campaign as campaign
from msi_autoencoder_wrapper.analysis.autoencoder.experiments import predictive_precompute as cache
from msi_autoencoder_wrapper.analysis.autoencoder.experiments import predictive_reports as reports
from msi_autoencoder_wrapper.visualization import predictive as plots

current = Path.cwd().resolve()
repository_root = next(path for path in (current, *current.parents) if (path / "pyproject.toml").is_file())
notebook_directory = repository_root / "assets/experiments/autoencoder_architecture/notebooks/07_09_26_predictive_expanded"
settings_path = Path(os.environ.get("MSI_PREDICTIVE_SETTINGS", notebook_directory / "analysis_settings.yaml"))
settings = campaign.load_settings(settings_path)
RESULTS = notebook_directory / "part_2_shared_inference_results"
RESULTS.mkdir(parents=True, exist_ok=True)


2026-09-08 10:25:30,325 | INFO     | msi_autoencoder_wrapper.utils.module_search:79 | Discovered 1 implementation module(s) in package 'msi_autoencoder_wrapper.binners.binners_strategies'.


2026-09-08 10:25:30,328 | INFO     | msi_autoencoder_wrapper.utils.module_search:79 | Discovered 6 implementation module(s) in package 'msi_autoencoder_wrapper.binners.inverse_strategies'.


2026-09-08 10:25:30,661 | INFO     | msi_autoencoder_wrapper.utils.module_search:79 | Discovered 3 implementation module(s) in package 'msi_autoencoder_wrapper.models.architectures.types.autoencoders.presets'.


2026-09-08 10:25:30,663 | INFO     | msi_autoencoder_wrapper.utils.module_search:79 | Discovered 23 implementation module(s) in package 'msi_autoencoder_wrapper.models.architectures.types.autoencoders'.


2026-09-08 10:25:30,664 | INFO     | msi_autoencoder_wrapper.utils.module_search:79 | Discovered 0 implementation module(s) in package 'msi_autoencoder_wrapper.models.architectures.types.schema'.


2026-09-08 10:25:30,665 | INFO     | msi_autoencoder_wrapper.utils.module_search:79 | Discovered 30 implementation module(s) in package 'msi_autoencoder_wrapper.models.architectures'.


## Restoring data and executing the campaign

### Methodology

#### Theoretical

Comparisons require identical spectra, class order and preprocessing. Full train/validation/test partitions are retained at pixel_fraction=1; geometry_sample_size limits only the cost of pairwise geometry.

#### Implementation

The dataset is restored from the saved configuration through the public loader and a temporary copy with local paths. The split is not regenerated from its seed. Batched inference runs in eval/no_grad and reads only the active head. Hashes cover weights, configuration, input tensor contents, the ion catalogue, settings, library sources and versions. The complete marker is written after the tables; rerunning reuses completed models.

#### Figure descriptions

The tables show campaign availability and models admitted to inference. sample_indices.csv records the realized sample positions. This section does not generate a figure.


In [2]:
models, sources = campaign.inventory(settings)
display(sources)
if not sources.loc[sources.required, "available"].all():
    raise FileNotFoundError("Download required campaigns or correct status_directory in analysis_settings.yaml; see README.md.")
display(models[["model_id", "label", "status", "ready", "duplicate_seed_condition"]])
import torch
if settings["device"] == "cpu":
    torch.set_num_threads(1)
run_directory = cache.precompute(models, settings)
reports.save_tables(RESULTS, inventory=models, samples=pd.read_csv(run_directory / "sample_indices.csv"), classes=pd.read_csv(run_directory / "classes.csv"))
(RESULTS / "cache_location.txt").write_text(str(run_directory))
print("Completed cache:", run_directory)

2026-09-08 10:25:35,666 | INFO     | msi_autoencoder_wrapper.analysis.autoencoder.experiments.predictive_campaign:250 | Audited 65 model manifests across 2 sources.


,source,required,status_directory,available
0,predictive_initial,True,/home/max/repositories/MSIAutoEncoderWrapper/d...,True
1,historical_bce,True,/home/max/repositories/MSIAutoEncoderWrapper/d...,True


,model_id,label,status,ready,duplicate_seed_condition
0,predictive_initial/task_000000,bce_global_positive_penalty (PositiveWeightedM...,completed,True,False
1,predictive_initial/task_000001,bce_global_positive_penalty (PositiveWeightedM...,completed,True,False
2,predictive_initial/task_000002,bce_global_positive_penalty (PositiveWeightedM...,completed,True,False
3,predictive_initial/task_000003,bce_global_positive_penalty (PositiveWeightedM...,completed,True,False
4,predictive_initial/task_000004,bce_global_positive_penalty (PositiveWeightedM...,completed,True,False
...,...,...,...,...,...
60,historical_bce/task_000000,balanced_bce (ClassBalancedMultiLabelBCELoss),completed,True,False
61,historical_bce/task_000001,balanced_bce (ClassBalancedMultiLabelBCELoss),completed,True,False
62,historical_bce/task_000002,balanced_bce (ClassBalancedMultiLabelBCELoss),completed,True,False
63,historical_bce/task_000003,balanced_bce (ClassBalancedMultiLabelBCELoss),completed,True,False


2026-09-08 10:25:35,742 | INFO     | msi_autoencoder_wrapper.utils.module_search:79 | Discovered 2 implementation module(s) in package 'msi_autoencoder_wrapper.models.datasets.strategies'.


2026-09-08 10:25:35,751 | INFO     | msi_autoencoder_wrapper.utils.module_search:79 | Discovered 21 implementation module(s) in package 'msi_autoencoder_wrapper.training.criterions.autoencoder'.


...

2026-09-08 10:26:37,715 | INFO     | msi_autoencoder_wrapper.analysis.autoencoder.experiments.predictive_precompute:245 | Reusing completed inference for historical_bce/task_000003.


2026-09-08 10:26:37,722 | INFO     | msi_autoencoder_wrapper.analysis.autoencoder.experiments.predictive_precompute:245 | Reusing completed inference for historical_bce/task_000004.


2026-09-08 10:26:37,724 | INFO     | msi_autoencoder_wrapper.analysis.autoencoder.experiments.predictive_precompute:304 | Completed analysis cache: /home/max/repositories/MSIAutoEncoderWrapper/data/kidney_workspace/cache/predictive_heads_analysis/evaluations/a48e49ffb8d61e7e3f1926e791332ca73c50c3005f79bc6917f27237fb44ca82


2026-09-08 10:26:37,798 | INFO     | msi_autoencoder_wrapper.analysis.autoencoder.experiments.predictive_reports:198 | Saved 3 analytical tables in /home/max/repositories/MSIAutoEncoderWrapper/assets/experiments/autoencoder_architecture/notebooks/07_09_26_predictive_expanded/part_2_shared_inference_results.


Completed cache: /home/max/repositories/MSIAutoEncoderWrapper/data/kidney_workspace/cache/predictive_heads_analysis/evaluations/a48e49ffb8d61e7e3f1926e791332ca73c50c3005f79bc6917f27237fb44ca82


### Remarks

Models with failed/running status are not evaluated. Available completed models can be analysed as a partial campaign, but incomplete conditions are excluded from the final shortlist. A missing required source ends this part with an explicit message.

### Notes


## Results / Summary

### LLM

An executable protocol and table exports have been prepared. Results from the target campaign are not yet available; this file does not contain an empirical conclusion that any head is superior. After execution, assess effect direction and magnitude, variation across seeds, completeness of pairs and limitations of the label definitions.

### Person
